# OpenAlex metadata extraction

In [1]:
import csv
import time
import requests
from pathlib import Path

import pandas as pd

from routehunter_build.extract import (resolve_source_id, 
                                       reconstruct_abstract, 
                                       fetch_works_for_source_year, 
                                       extract_row, 
                                       get_year_counts)

In [2]:
SLEEP_BETWEEN_REQUESTS = 0.1  # stay comfortably under 100 req/sec

JOURNAL_NAMES = [
    'Natural Product Reports',
    'Chemistry - A European Journal',
    'Advanced Synthesis & Catalysis',
    'Organic & Biomolecular Chemistry',
    'Beilstein Journal of Organic Chemistry',
    'ACS Catalysis',
    'Chemical Communications',
    'RSC Advances',
    'New Journal of Chemistry',
    'Heterocycles',
    'Chemistry of Heterocyclic Compounds',
    'Chinese Journal of Chemistry',
    'Helvetica Chimica Acta',
    'Molecules',
]

### Step 1 - resolve journal names

In [3]:
sources = {}
for name in JOURNAL_NAMES:
    record = resolve_source_id(name, verbose=False)
    if record:
        sources[name] = record["id"]
    time.sleep(SLEEP_BETWEEN_REQUESTS)
print(f"Resolved {len(sources)}/{len(JOURNAL_NAMES)} journals.")
sources

Resolved 14/14 journals.


{'Natural Product Reports': 'https://openalex.org/S18855311',
 'Chemistry - A European Journal': 'https://openalex.org/S68911691',
 'Advanced Synthesis & Catalysis': 'https://openalex.org/S36069537',
 'Organic & Biomolecular Chemistry': 'https://openalex.org/S179810352',
 'Beilstein Journal of Organic Chemistry': 'https://openalex.org/S13215905',
 'ACS Catalysis': 'https://openalex.org/S37391459',
 'Chemical Communications': 'https://openalex.org/S152760256',
 'RSC Advances': 'https://openalex.org/S2481244646',
 'New Journal of Chemistry': 'https://openalex.org/S36793815',
 'Heterocycles': 'https://openalex.org/S27298842',
 'Chemistry of Heterocyclic Compounds': 'https://openalex.org/S110436767',
 'Chinese Journal of Chemistry': 'https://openalex.org/S112006741',
 'Helvetica Chimica Acta': 'https://openalex.org/S181847678',
 'Molecules': 'https://openalex.org/S108911230'}

### Step 2 - count the number of papers per year/journal

In [4]:
counts = {journal: get_year_counts(source_id) for journal, source_id in sources.items()}

In [5]:
df_counts = pd.DataFrame(counts).sort_index()
df_counts.index.name = "year"
df_counts = df_counts.fillna(0).astype(int)
df_counts.sum(axis=0).sort_values(ascending=False)

RSC Advances                              83684
Chemical Communications                   78903
Molecules                                 65840
Chemistry - A European Journal            53280
New Journal of Chemistry                  30875
Helvetica Chimica Acta                    26120
Organic & Biomolecular Chemistry          25637
Chemistry of Heterocyclic Compounds       18425
Heterocycles                              17897
ACS Catalysis                             17814
Advanced Synthesis & Catalysis            12342
Chinese Journal of Chemistry              10698
Beilstein Journal of Organic Chemistry     4345
Natural Product Reports                    3582
dtype: int64

## Step 3 - full extraction (all journals).

In [6]:
# sources_to_extract = {
#     'Synthetic Communications': sources['Synthetic Communications'],         
#     'Journal of Synthetic Organic Chemistry, Japan': sources['Journal of Synthetic Organic Chemistry, Japan']
# }
sources_to_extract = sources

In [8]:
min_list, max_list = [], []
for col in sources_to_extract.keys():
    min_list.append(df_counts[df_counts[col] != 0][col].index.min())
min(sorted(min_list))

'1918'

In [9]:
OUTPUT_CSV = "openalex_chem_metadata_tmp.csv"
YEAR_START, YEAR_END = 1918, 2025

In [10]:
fieldnames = [
    "journal", "openalex_id", "doi", "title", "year",
    "publication_date", "authors", "cited_by_count", "is_oa", "abstract",
]

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

    for journal_name, source_id in sources_to_extract.items():
        for year in range(YEAR_START, YEAR_END + 1):
            count = 0
            for work in fetch_works_for_source_year(source_id, year):
                writer.writerow(extract_row(work, journal_name))
                count += 1
            print(f"  {journal_name} / {year}: {count} works")
print(f"\nDone. Saved to {OUTPUT_CSV}")

  Natural Product Reports / 1918: 0 works
  Natural Product Reports / 1919: 0 works
  Natural Product Reports / 1920: 0 works
  Natural Product Reports / 1921: 0 works
  Natural Product Reports / 1922: 0 works
  Natural Product Reports / 1923: 0 works
  Natural Product Reports / 1924: 0 works
  Natural Product Reports / 1925: 0 works
  Natural Product Reports / 1926: 0 works
  Natural Product Reports / 1927: 0 works
  Natural Product Reports / 1928: 0 works
  Natural Product Reports / 1929: 0 works
  Natural Product Reports / 1930: 0 works
  Natural Product Reports / 1931: 0 works
  Natural Product Reports / 1932: 0 works
  Natural Product Reports / 1933: 0 works
  Natural Product Reports / 1934: 0 works
  Natural Product Reports / 1935: 0 works
  Natural Product Reports / 1936: 0 works
  Natural Product Reports / 1937: 0 works
  Natural Product Reports / 1938: 0 works
  Natural Product Reports / 1939: 0 works
  Natural Product Reports / 1940: 0 works
  Natural Product Reports / 1941: 

## Step 4 - final file preparation

In [11]:
import pandas as pd

all_data = pd.read_csv("openalex_chem_metadata_tmp.csv")
print(f"Total number of papers: {len(all_data)}")
all_data.isna().sum()

Total number of papers: 433374


journal                 0
openalex_id             0
doi                  2992
title                3202
year                    0
publication_date        0
authors             21979
cited_by_count          0
is_oa                   0
abstract            39951
dtype: int64

In [12]:
final_data = all_data.copy()
final_data = final_data[["journal", "title", "abstract", "doi", "publication_date", "cited_by_count"]]
final_data["doi"] = [str(i).replace("https://doi.org/", "") for i in final_data["doi"]]
final_data = final_data.dropna(subset="title")
final_data.to_csv("openalex_chem_metadata_3.csv", index=False)